In [ ]:
import pandas as pd

file_path = "../data/raw/games.csv"

# 원본 header 확인
original_columns = pd.read_csv(file_path, nrows=0).columns.tolist()

# 잘못 합쳐진 header를 두 개로 분리
corrected_columns = []

for col in original_columns:
    if col == "DiscountDLC count":
        corrected_columns.extend(["Discount", "DLC count"])
    else:
        corrected_columns.append(col)

# 올바른 40개 컬럼명으로 다시 로드
df = pd.read_csv(
    file_path,
    header=0,
    names=corrected_columns,
    index_col=False
)

df.shape

In [ ]:
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df[
    [
        "AppID",
        "Name",
        "Release date",
        "Estimated owners",
        "Peak CCU",
        "Required age",
        "Price",
        "Discount",
        "DLC count"
    ]
].head()

### CSV Parsing Issue

Initial inspection showed that the CSV contained 39 header names while each data row contained 40 fields.

The header `DiscountDLC count` represented two separate variables:

- Discount
- DLC count

As a result, Pandas initially interpreted the actual AppID values as the DataFrame index and shifted all remaining fields by one column.

The raw CSV was not modified. Instead, the file was reloaded using a corrected 40-column schema.

In [ ]:
print("Shape:", df.shape)
print("Total AppID:", len(df["AppID"]))
print("Unique AppID:", df["AppID"].nunique())
print("Missing AppID:", df["AppID"].isna().sum())
print("Duplicated AppID:", df["AppID"].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

### Dataset Grain & Key Validation

After correcting the CSV header issue, the dataset contains:

- 125,855 rows
- 40 columns
- 125,855 unique AppIDs
- 0 missing AppIDs
- 0 duplicated AppIDs
- 0 fully duplicated rows

This confirms that each row represents one unique Steam AppID record.

However, a unique Steam AppID does not necessarily mean that every record
represents a standard released game. Items such as playtests or other
Steam application types may still be present and should be considered
during further data auditing.

In [ ]:
df.isna().sum()

In [ ]:
missing_count = df.isna().sum()

missing_count[missing_count > 0].sort_values(ascending=False)

In [ ]:
# 1차 분석에서 명확히 제외할 컬럼
columns_to_drop_phase1 = [
    "About the game",
    "Full audio languages",
    "Header image",
    "Website",
    "Support url",
    "Support email",
    "Metacritic url",
    "Notes",
    "Screenshots",
    "Movies"
]

df_analysis = df.drop(columns=columns_to_drop_phase1)

In [ ]:
df_analysis.shape

In [ ]:
df_analysis.columns.tolist()

### Phase 1 Column Selection

Columns with no direct relevance to the current business questions were
excluded from the working analysis DataFrame.

Potentially useful secondary variables were retained until their coverage,
value structure, and analytical relevance can be evaluated.

The corrected raw DataFrame (`df`) remains unchanged.

In [ ]:
##각 컬럼에서 전체 125,855행 중 몇 %가 결측인지 나옴
missing_count = df_analysis.isna().sum()
missing_rate = (missing_count / len(df_analysis)) * 100

missing_rate[missing_rate > 0].sort_values(ascending=False)

In [ ]:
missing_audit = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Rate (%)": missing_rate
})

missing_audit = missing_audit[
    missing_audit["Missing Count"] > 0
].sort_values("Missing Rate (%)", ascending=False)

missing_audit

In [ ]:
df_analysis[df_analysis["Name"].isna()]

In [ ]:
df_analysis["Reviews"].dropna().head(10)

In [ ]:
# ---------------------------------------
# Missing Value Audit - Cleaning Decisions
# ---------------------------------------

# 1. 분석 정보가 거의 없는 Name 결측 레코드 제외
df_analysis = df_analysis[df_analysis["Name"].notna()].copy()

# 2. Missing coverage 및 데이터 의미를 검토한 결과
#    현재 분석 범위에서 사용할 수 없는 컬럼 제외
columns_to_drop_phase2 = [
    "Score rank",
    "Reviews"
]

df_analysis = df_analysis.drop(columns=columns_to_drop_phase2)

# 결과 확인
df_analysis.shape

In [ ]:
print("Rows:", len(df_analysis))
print("Columns:", len(df_analysis.columns))
print("Missing Name:", df_analysis["Name"].isna().sum())

In [ ]:
[col for col in ["Reviews", "Score rank"] if col in df_analysis.columns]

### Missing Value Audit — Summary

Missing values were evaluated based on their analytical relevance rather than 
using a single global rule such as dropping all incomplete rows.

The audit identified several different types of missingness:

- `Score rank` is missing for approximately 99.97% of records. Its coverage is
  insufficient for meaningful market-level comparison, so it was excluded
  from the working analysis dataset.

- `Reviews` is missing for approximately 90.32% of records. Inspection of
  non-missing values showed that this field contains free-text review excerpts
  rather than review-volume data. Since NLP/text analysis is outside the
  current project scope, the column was excluded.

- `Tags` has relatively high missingness (approximately 33.77%) but remains
  potentially useful for later niche-level drill-down analysis. It was
  therefore retained as a secondary variable.

- `Categories`, `Developers`, and `Publishers` have approximately 7% missing
  values. These variables were retained because they may support secondary
  segmentation or diagnostic analysis.

- `Genres` is missing for approximately 6.69% of records. Because Genres is a
  core analytical dimension across several business questions, these records
  will not be removed globally. Instead, genre-missing records will be handled
  specifically in analyses that require genre information.

One record (`AppID 396420`) had a missing `Name` and contained almost no usable
analytical information: most metrics were zero and several descriptive fields,
including Developers, Publishers, Categories, Genres, and Tags, were missing.
This record was excluded from the working analysis dataset because it provides
insufficient information for meaningful market analysis and could otherwise
distort record counts.

Missing-value treatment therefore remains analysis-specific wherever possible,
rather than removing complete records solely because one variable is missing.

---
### Zero Value Audit

In [ ]:
numeric_columns = df_analysis.select_dtypes(include="number").columns

numeric_columns

In [ ]:
zero_count = (df_analysis[numeric_columns] == 0).sum()  # Zero Count
zero_rate = (zero_count / len(df_analysis)) * 100  # Zero Rate (%)

In [ ]:
zero_audit = pd.DataFrame({
    "Zero Count": zero_count,
    "Zero Rate (%)": zero_rate
})

zero_audit = zero_audit[
    zero_audit["Zero Count"] > 0
].sort_values("Zero Rate (%)", ascending=False)

zero_audit

In [ ]:
df_analysis["User score"].value_counts().head(10)

In [ ]:
df_analysis["Metacritic score"].value_counts().head(10)

In [ ]:
df_analysis["Peak CCU"].describe()

In [ ]:
df_analysis["Recommendations"].describe()

In [ ]:
df_analysis[
    [
        "Average playtime forever",
        "Median playtime forever",
        "Average playtime two weeks",
        "Median playtime two weeks"
    ]
].describe()

In [ ]:
((df_analysis["Positive"] == 0) & (df_analysis["Negative"] == 0)).sum()

In [ ]:
df_analysis.loc[df_analysis["Peak CCU"] > 0, "Peak CCU"].describe()

Peak CCU will be retained as a traction/activity proxy, but its highly zero-inflated and right-skewed distribution means that mean values alone should not be used for market comparisons.

In [ ]:
df_analysis.loc[
    df_analysis["Recommendations"] > 0,
    "Recommendations"
].describe()

In [ ]:
df_analysis.loc[
    df_analysis["Average playtime forever"] > 0,
    "Average playtime forever"
].describe()

In [ ]:
df_analysis.loc[
    df_analysis["Recommendations"] > 0,
    "Recommendations"
].sort_values().head(20)

In [ ]:
# 정말 1~100이 하나도 없는지 확인
((df_analysis["Recommendations"] > 0) &
 (df_analysis["Recommendations"] <= 100)).sum()

In [ ]:
# 6.5년에 해당하는 플레이타임 확인
df_analysis.nlargest(
    10,
    "Average playtime forever"
)[
    [
        "AppID",
        "Name",
        "Release date",
        "Estimated owners",
        "Average playtime forever",
        "Median playtime forever"
    ]
]

### Zero Value Audit - Summary

Zero values were not treated uniformly because their meaning differs across variables.

- `User score` is zero for approximately 99.97% of records and therefore provides
  insufficient coverage for market-level comparison. It will not be used as a core
  satisfaction metric.

- `Metacritic score` is zero for approximately 96.6% of records and will only be
  considered as a secondary metric for the subset of games with available scores.

- `Peak CCU` is highly zero-inflated and strongly right-skewed. Among non-zero values,
  the median is 3 while the mean is approximately 340, indicating that a small number
  of games strongly influence the average. It will be retained as a traction/activity
  proxy, but mean values will not be used alone.

- `Recommendations` is also highly zero-inflated and right-skewed. No values between
  1 and 100 were observed; the smallest non-zero value is 101. The scraper stores the
  Steam recommendation total when available and otherwise stores 0, so zero values
  should not automatically be interpreted as exactly zero recommendations.
  This variable will be retained with this limitation documented.

- Lifetime playtime variables are highly zero-inflated and contain extreme values.
  They remain candidate engagement proxies, but extreme observations require further
  investigation during the Numeric Distribution Audit.

- Two-week playtime variables are zero for approximately 96.9% of records and will
  therefore be treated as secondary recent-activity indicators rather than core
  engagement metrics.

- `Positive` and `Negative` are both zero for 42,898 records. Satisfaction ratios
  cannot be calculated for these records because the denominator would be zero.

- Zero values in variables such as `Price`, `Discount`, `DLC count`, `Achievements`,
  and `Required age` may represent valid real-world states and will not be converted
  to missing values without additional analytical justification.

### 5. Categorical Structure Audit

In [ ]:
categorical_columns = [
    "Genres",
    "Categories",
    "Tags",
    "Developers",
    "Publishers",
    "Supported languages"
]

In [ ]:
df_analysis[categorical_columns].dtypes

In [ ]:
for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df_analysis[col].dropna().head(5).tolist())

In [ ]:
for col in categorical_columns:
    value = df_analysis[col].dropna().iloc[0]
    print(col, type(value), repr(value))

In [ ]:
for col in categorical_columns:
    print(col, df_analysis[col].nunique())

In [ ]:
# 각 컬럼이 실제로 얼마나 multi-value인지 확인
multi_value_columns = [
    "Genres",
    "Categories",
    "Tags",
    "Developers",
    "Publishers"
]

for col in multi_value_columns:
    multi_count = df_analysis[col].dropna().str.contains(",", regex=False).sum()
    total = df_analysis[col].notna().sum()
    
    print(
        col,
        "Multi-value rows:",
        multi_count,
        f"({multi_count / total * 100:.2f}%)"
    )

In [ ]:
# Developers, Publishers 실제 복수값 사례
for col in ["Developers", "Publishers"]:
    print(f"\n--- {col}: multi-value examples ---")
    
    print(
        df_analysis.loc[
            df_analysis[col].fillna("").str.contains(",", regex=False),
            col
        ].head(10).tolist()
    )

In [ ]:
# Supported languages 구조 확인
df_analysis["Supported languages"].value_counts().head(10)

In [ ]:
# missing이 아니라 문자열로 저장된 빈 리스트
(df_analysis["Supported languages"] == "[]").sum()

### Categorical Structure Audit — Summary

Several categorical variables contain multiple values within a single record.

- `Genres`, `Categories`, and `Tags` are stored as comma-separated strings.
  Multi-value records account for approximately 85.65%, 95.56%, and 99.35%
  of their non-missing observations respectively.

- The previously observed `nunique()` values therefore represent unique
  combinations of categories rather than the number of individual genres,
  categories, or tags.

- These fields will require parsing and, where appropriate, exploding before
  category-level market analysis.

- `Developers` and `Publishers` also contain commas, but commas are not always
  reliable delimiters because they may occur within organization names
  (e.g. "Accolade, Inc."). These fields will therefore not be split using a
  simple comma rule unless they become necessary for later analysis.

- `Supported languages` is stored as a string representation of a list rather
  than as an actual Python list. In addition, 8,390 records contain `"[]"`,
  indicating an empty language list despite the absence of NaN values.

Categorical transformations will therefore be column-specific rather than
applying one generic parsing rule to all multi-value fields.

---
### 6. Numeric Distribution Audit

In [ ]:
numeric_audit_columns = [
    "Peak CCU",
    "Required age",
    "Price",
    "Discount",
    "DLC count",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Achievements",
    "Recommendations",
    "Average playtime forever",
    "Average playtime two weeks",
    "Median playtime forever",
    "Median playtime two weeks"
]

In [ ]:
# 전체 분포 요약 - 기본 describe()보다 95%, 99% 같은 상위 percentile을 같이 보는 게 좋음
numeric_summary = df_analysis[numeric_audit_columns].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

# .T는 표를 돌려서 컬럼 하나당 한 행으로 보기 쉽게 만드는 것

numeric_summary

In [ ]:
# Price 상위 레코드 확인 
df_analysis.nlargest(
    10,
    "Price"
)[
    ["AppID", "Name", "Release date", "Price", "Genres"]
]

In [ ]:
# DLC Count 상위 레코드 확인
df_analysis.nlargest(
    10,
    "DLC count"
)[
    ["AppID", "Name", "Release date", "DLC count", "Genres"]
]


In [ ]:
# Achievements 상위 레코드 확인
df_analysis.nlargest(
    10,
    "Achievements"
)[
    ["AppID", "Name", "Release date", "Achievements", "Genres"]
]

### Numeric Distribution Audit — Extreme Values

The numeric variables show strongly right-skewed distributions, with several
extreme observations.

Inspection of the highest values suggests that extreme values should not be
removed automatically.

- `Price` contains values close to 1,000. Inspection showed that some of these
  records represent software or other non-standard game products, revealing a
  broader dataset-scope issue rather than a simple price-data error.

- `DLC count` reaches 3,703. The highest values are associated with products
  such as Fantasy Grounds, RPG Maker, and Train Simulator, where very large
  content catalogs may be structurally plausible.

- `Achievements` reaches 9,821, with several records containing approximately
  5,000 achievements. These are highly atypical but cannot yet be classified
  as data errors.

- `Peak CCU`, review counts, recommendations, and playtime metrics also show
  extreme right-skewness. Mean values alone may therefore be misleading for
  market-level comparisons.

Extreme values will be retained during the audit stage and handled only when
their analytical impact and product context are understood.

### 6.1 Product Scope Audit

The numeric distribution audit identified several high-priced records that
appear to represent software or other non-standard game products.

Because the project aims to analyze the Steam game market for a small indie
game studio, the dataset scope should be checked before market-level analysis.

This step does not remove records. It only identifies potentially out-of-scope
Steam applications for later cleaning decisions.

In [ ]:
# Genres를 개별 genre 단위로 펼쳐서 실제 어떤 값들이 존재하는지 확인
genre_tokens = (
    df_analysis["Genres"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

genre_tokens.value_counts().head(50)

In [ ]:
possible_non_game_genres = [
    "Animation & Modeling",
    "Design & Illustration",
    "Utilities",
    "Video Production",
    "Audio Production",
    "Web Publishing",
    "Software Training",
    "Game Development",
    "Photo Editing"
]

In [ ]:
for genre in possible_non_game_genres:
    count = (
        df_analysis["Genres"]
        .fillna("")
        .str.contains(genre, regex=False)
        .sum()
    )

    print(genre, count)

In [ ]:
df_analysis[
    df_analysis["Genres"]
    .fillna("")
    .str.contains(
        "Animation & Modeling|Design & Illustration|Utilities|"
        "Video Production|Audio Production|Web Publishing|"
        "Software Training|Game Development|Photo Editing",
        regex=True
    )
][
    ["AppID", "Name", "Genres", "Price"]
].head(30)

In [ ]:
df_analysis[
    df_analysis["Name"]
    .fillna("")
    .str.contains("Playtest", case=False, regex=False)
][
    ["AppID", "Name", "Release date", "Genres", "Price"]
].head(20)

In [ ]:
df_analysis["Name"].fillna("").str.contains(
    "Playtest",
    case=False,
    regex=False
).sum()

### 6.1 Product Scope Audit — Summary

The dataset contains Steam applications that are not directly comparable to
standard released games.

The `Genres` field is not a pure game-genre taxonomy. Alongside conventional
genres such as Action, RPG, and Strategy, it also contains software-oriented
labels such as Utilities, Animation & Modeling, Design & Illustration,
Software Training, and Game Development.

These labels cannot be used as automatic exclusion criteria because some
records combine software-related labels with conventional game genres.
Potential non-game applications will therefore be flagged rather than removed
solely based on genre.

A separate issue was identified for Playtest applications. 8,139 records contain
"Playtest" in the application name, and inspected examples frequently have no
genre information and a price of zero. These records are not directly comparable
to standard released products and are strong candidates for exclusion from the
main game-market analysis dataset.

The final product-scope filtering rule will be applied during the cleaning stage,
not during the audit stage.

---
### 7. Date Coverage Audit

In [ ]:
# dtype check
df_analysis["Release date"].dtype

In [ ]:
release_date_parsed = pd.to_datetime(
    df_analysis["Release date"],
    errors="coerce"
)

In [ ]:
print("Earliest date:", release_date_parsed.min())
print("Latest date:", release_date_parsed.max())
print("Unparseable dates:", release_date_parsed.isna().sum())

In [ ]:
release_date_parsed.dt.year.value_counts().sort_index()

In [ ]:
analysis_date = pd.Timestamp("2026-09-22")

future_release_mask = release_date_parsed > analysis_date

print("Future-dated records:", future_release_mask.sum())
print(
    "Released / current-date records:",
    (~future_release_mask).sum()
)

In [ ]:
df_analysis.loc[
    future_release_mask,
    [
        "AppID",
        "Name",
        "Release date",
        "Estimated owners",
        "Price",
        "Genres"
    ]
].head(20)

### Date Coverage Audit — Summary

`Release date` is stored as a string in the raw dataset, but all 125,854
records were successfully parsed as datetime values.

The observed release-date range is:

- Earliest: 1997-06-30
- Latest: 2026-12-01

Only one record has a release date later than the analysis date
(2026-09-22). Therefore, future-dated applications are not a substantial
coverage issue in this dataset.

However, 2026 remains an incomplete calendar year. Annual release counts
for 2026 should therefore not be directly compared with complete years
such as 2024 or 2025.

The historical distribution is also highly uneven across years, so changes
in raw annual record counts should not automatically be interpreted as
market growth without considering dataset coverage and Steam catalog
development.

---
### 8. Estimated Owners Structure

In [ ]:
df_analysis["Estimated owners"].dtype

In [ ]:
df_analysis["Estimated owners"].nunique()

In [ ]:
# 실제 범위와 빈도
owners_counts = df_analysis["Estimated owners"].value_counts()

owners_counts

In [ ]:
# 형식 검증 - 0이면 모든 값이 동일한 range 형식을 따른다는 의미
owners_pattern = r"^\d+\s*-\s*\d+$"

invalid_owners = ~df_analysis["Estimated owners"].str.match(
    owners_pattern,
    na=False
)

print("Invalid format:", invalid_owners.sum())

In [ ]:
# audit용으로만 lower / upper bound 확인, df_analysis 에 컬럼 추가 안 함
owners_bounds = (
    df_analysis["Estimated owners"]
    .str.split("-", expand=True)
    .rename(columns={0: "Lower", 1: "Upper"})
)

owners_bounds["Lower"] = owners_bounds["Lower"].str.strip().astype(int)
owners_bounds["Upper"] = owners_bounds["Upper"].str.strip().astype(int)

owners_bounds.head()

In [ ]:
(owners_bounds["Lower"] > owners_bounds["Upper"]).sum()

In [ ]:
# 0-0 구간 확인
zero_zero_count = (
    df_analysis["Estimated owners"] == "0 - 0"
).sum()

zero_zero_rate = zero_zero_count / len(df_analysis) * 100

print("0 - 0 count:", zero_zero_count)
print(f"0 - 0 rate: {zero_zero_rate:.2f}%")

In [ ]:
# 구간 폭이 일정한지 확인
owners_bounds["Width"] = (
    owners_bounds["Upper"] - owners_bounds["Lower"]
)

owners_bounds["Width"].value_counts().sort_index()

In [ ]:
# 0-0 게임 중 다른 traction 신호가 있는지 확인
zero_owner_mask = df_analysis["Estimated owners"] == "0 - 0"

print("0-0 total:", zero_owner_mask.sum())

print(
    "0-0 with Positive > 0:",
    (zero_owner_mask & (df_analysis["Positive"] > 0)).sum()
)

print(
    "0-0 with Negative > 0:",
    (zero_owner_mask & (df_analysis["Negative"] > 0)).sum()
)

print(
    "0-0 with Peak CCU > 0:",
    (zero_owner_mask & (df_analysis["Peak CCU"] > 0)).sum()
)

print(
    "0-0 with Recommendations > 0:",
    (zero_owner_mask & (df_analysis["Recommendations"] > 0)).sum()
)

In [ ]:
df_analysis.loc[
    zero_owner_mask &
    (
        (df_analysis["Positive"] > 0) |
        (df_analysis["Peak CCU"] > 0) |
        (df_analysis["Recommendations"] > 0)
    ),
    [
        "AppID",
        "Name",
        "Estimated owners",
        "Peak CCU",
        "Positive",
        "Negative",
        "Recommendations",
        "Genres"
    ]
].head(20)

### Estimated Owners Structure Audit — Summary

`Estimated owners` is stored as a string-based interval rather than a
continuous numeric variable.

The dataset contains 14 distinct owner ranges. All records follow a valid
`lower - upper` format, and no records contain a lower bound greater than
their upper bound.

The intervals are not equally sized, so the variable should not be treated
as a standard continuous numeric measure without additional assumptions.

A notable category is `0 - 0`, which appears in 24,578 records
(approximately 19.53% of the analysis dataset).

Cross-checking these records against other traction indicators showed that
372 records with `Estimated owners = 0 - 0` still have non-zero
`Recommendations`. Therefore, `0 - 0` cannot safely be interpreted as
confirmed zero ownership.

The `0 - 0` range will be treated as a separate uncertain/unavailable
ownership state rather than converted to numeric zero.

For subsequent analysis, the original owner ranges will be preserved.
Ordered owner bands and lower/upper bounds may be derived where analytically
useful, but midpoint values will not be presented as actual owner counts.

Estimated owners will only be used as a traction proxy and will not be
interpreted as sales or revenue.

---
## 9. Audit Log / Decisions

The data audit was conducted before cleaning according to the principle:

**Raw → Audit → Decision → Clean / Transform → Validate → Processed**

The raw CSV remains unchanged. All cleaning and transformation decisions will
be implemented separately in `02_data_cleaning.ipynb`.

### Dataset Status

After correcting the CSV header misalignment:

- Rows: 125,855
- Columns: 40
- Unique AppIDs: 125,855
- Missing AppIDs: 0
- Duplicated AppIDs: 0
- Fully duplicated rows: 0

One analytically unusable record (`AppID 396420`) was identified and excluded
from the working analysis dataset.

The current working DataFrame contains:

- 125,854 records
- 28 retained candidate columns


---
### Final Audit Log

| Area / Column | Issue observed | Business relevance | Decision for cleaning | Reason |
|---|---|---|---|---|
| CSV structure | `Discount` and `DLC count` merged in header, causing 39 headers for 40 fields | Entire project | Correct schema on import | Raw values were shifted under incorrect headers |
| AppID | No missing or duplicate IDs after correcting import | All questions | Keep | Reliable unique identifier |
| AppID 396420 | Missing name; most descriptive fields missing and metrics zero | Q1–Q6 | Exclude record | Insufficient analytical information |
| Metadata/media columns | Header image, URLs, screenshots, movies etc. | None for Q1–Q6 | Exclude | Outside analytical scope |
| Reviews | 90.32% missing; values are free-text review excerpts | Q2/Q3 initially | Exclude | Not review-volume data; NLP outside scope |
| Score rank | 99.97% missing | Satisfaction candidate | Exclude | Insufficient coverage |
| Genres | 6.69% missing; mostly multi-value strings | Q1–Q6 | Keep; parse later | Core market dimension |
| Categories | 7.12% missing; mostly multi-value | Q3/Q6 | Keep as secondary | Useful for drill-down |
| Tags | 33.77% missing; almost always multi-value | Q6 | Keep as secondary | Useful niche dimension but incomplete |
| Developers / Publishers | Missing values plus ambiguous comma delimiters | Secondary | Keep unparsed initially | Simple splitting could create false entities |
| Supported languages | Stringified list; 8,390 `[]` values | Secondary | Parse only if needed | Empty list represents unavailable information not captured as NaN |
| User score | ~99.97% zero | Q3 candidate | Exclude from core analysis | Effective coverage too low |
| Metacritic score | ~96.6% zero | Secondary satisfaction context | Keep secondary | Only usable for small covered subset |
| Peak CCU | ~84.4% zero; strongly right-skewed | Q2/Q4/Q6 | Keep with caution | Useful activity proxy; mean alone misleading |
| Recommendations | ~83.3% zero; non-zero values start at 101 | Q2/Q3/Q6 | Keep with caution | Useful signal, but zero cannot safely be interpreted as exact zero recommendations |
| Positive / Negative | Both zero for 42,898 records | Q3/Q4 | Keep | Satisfaction ratio only valid where `Positive + Negative > 0` |
| Lifetime playtime | ~79.2% zero; strong skew and extreme outliers | Q3 | Keep as engagement proxy | Distribution-aware analysis required |
| Two-week playtime | ~96.9% zero | Recent engagement | Secondary only | Coverage too sparse for primary comparison |
| Price | 21.18% zero; extreme values up to ~999.98 | Q4 | Keep | Zero may mean free-to-play; extreme values revealed product-scope issues |
| Discount | Many zeros | Q4 secondary | Keep | Zero is plausibly a valid “no discount” state |
| DLC count | Highly skewed; max 3,703 | Product feature | Keep secondary | Extreme values may be legitimate product structure |
| Achievements | Highly skewed; max 9,821 | Product feature | Keep secondary | Extreme but not confirmed erroneous |
| Product scope | Software-like Steam applications present | Q1/Q4 especially | Flag, not blanket-drop | Software labels may coexist with game genres |
| Playtest | 8,139 names contain `Playtest` | Q1–Q6 | Exclusion candidate for main game-market dataset | Not directly comparable with standard released games |
| Release date | Stored as string; 100% parseable | Q1/Q5 and controls | Convert to datetime | Required for time analysis |
| Future release | Only one record after 2026-09-22 | Mostly traction/trend | Filter where relevant | Negligible overall, but no observed traction period |
| 2026 | Partial year | Q5 | Do not directly compare with complete years | Incomplete calendar-year coverage |
| Historical dates | Very uneven record counts across years | Q5 | Define trend window later | Raw annual counts do not automatically equal market growth |
| Estimated owners | 14 non-uniform string ranges | Q2/Q4/Q5/Q6 | Preserve original range; derive bounds if needed | Not an exact numeric owner count |
| Estimated owners `0 - 0` | 24,578 records (19.53%); 372 still have Recommendations > 0 | Q2/Q4/Q5/Q6 | Treat as uncertain/unavailable state | Cannot safely mean confirmed zero owners |
| Estimated owners midpoint | Unequal interval widths | Q2/Q4/Q5/Q6 | Do not treat midpoint as actual owners | Would create false precision |